<a href="https://colab.research.google.com/github/andilMc/gemmafro-e2b/blob/main/notebooks/07_demo_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Démo live Gradio (Gemma E2B + LoRA)

Interface de test en direct

In [ ]:
# Installe les dépendances (gradio pour l'interface, le reste comme 04/05).
!pip install -q -U transformers accelerate peft bitsandbytes gradio

In [ ]:
# Monte Drive et localise l'adaptateur LoRA final produit en Phase 3.
import os
from google.colab import drive

drive.mount('/content/drive')  # demande l'autorisation d'accès au Drive

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'  # racine du projet sur Drive
ADAPTER_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'  # adaptateur LoRA final de la Phase 3
assert os.path.isdir(ADAPTER_DIR), "Adaptateur introuvable — exécuter 03_finetune.ipynb jusqu'au bout d'abord."

In [ ]:
# Se connecte à Hugging Face avec le token des Colab Secrets.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))  # authentifie la session avec le token HF_TOKEN

In [ ]:
# Charge le modèle de base en 4-bit et lui rattache l'adaptateur LoRA — identique à 04_evaluate.ipynb et
# 05_generate_submission.ipynb, pour garantir exactement le même comportement que ce qui a été évalué.
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "google/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512  # même longueur maximale qu'à l'entraînement

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer Gemma
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # pas de token de padding défini : on réutilise le token de fin

gc.collect()  # libère la mémoire (Python, puis cache GPU)
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(  # même quantification 4-bit qu'à l'entraînement
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(  # charge le modèle de base quantifié
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},  # tout sur le GPU 0, sans offload CPU
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = True  # inférence : le cache KV accélère la génération

def unwrap_clippable_linears(model):
    """Même déballage qu'en Phase 3/4/5 : nécessaire pour que PeftModel retrouve la structure
    de modules sur laquelle l'adaptateur a été entraîné."""
    count = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if child.__class__.__name__ == "Gemma4ClippableLinear":
                setattr(module, child_name, child.linear)
                count += 1
    print(f'{count} couches Gemma4ClippableLinear déballées')
    return model

base_model = unwrap_clippable_linears(base_model)  # déballage avant de charger l'adaptateur

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)  # rattache les poids LoRA au modèle de base
model.eval()  # mode inférence (désactive le dropout)

gc.collect()
torch.cuda.empty_cache()
print(f"Mémoire GPU allouée : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

In [ ]:
# Même template de prompt qu'en Phases 2/3/4/5, mais avec le nom de langue choisi directement dans
# l'interface (pas besoin du code subset ici, puisqu'on ne travaille plus par sous-ensemble Val/Test).
LANGUAGES = ['English', 'Amharic', 'Luganda', 'Swahili', 'Akan']  # dans l'ordre du menu déroulant

def build_prompt(question: str, language: str) -> str:
    return (
        f"Réponds à la question de santé suivante en {language}, "
        f"de façon claire et médicalement fiable.\n\nQuestion : {question}"
    )

In [ ]:
# Génération en streaming (mot par mot), via TextIteratorStreamer dans un thread séparé.
# Sans ça, l'utilisateur ne voit rien bouger pendant les ~20-45s que prend une génération sur ce
# GPU (T4/L4 partagé) — ce qui donnait l'impression que le bouton ne faisait rien, alors que la
# démo fonctionnait déjà correctement (juste sans aucun retour visuel avant la fin).
import time
from threading import Thread
from transformers import TextIteratorStreamer

def stream_one(chat_prompt, max_new_tokens=400):
    """Génère token par token et cède (yield) le texte accumulé au fur et à mesure — c'est ce
    qui permet à Gradio d'afficher la réponse progressivement plutôt que d'un bloc à la fin."""
    inputs = tokenizer(chat_prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, streamer=streamer,
    )

    def _run():
        with torch.no_grad():
            model.generate(**gen_kwargs)

    thread = Thread(target=_run)
    thread.start()

    partial = ""
    for new_text in streamer:
        partial += new_text
        yield partial
    thread.join()

In [ ]:
# Interface Gradio : question + langue en entrée, réponse fine-tunée (+ zero-shot en option) en sortie.
#
# Les deux réponses ne peuvent pas être générées en parallèle au sens strict : elles partagent le
# même modèle chargé une seule fois en mémoire, et le zero-shot repose sur `model.disable_adapter()`,
# un bascule global sur cet objet partagé — l'activer pendant qu'un autre thread génère avec
# l'adaptateur activé corromprait les deux générations. Nous avons donc gardé un GPU unique
# (le seul disponible sur Colab) et un ordre séquentiel, mais chaque réponse s'affiche
# indépendamment et progressivement dès qu'elle est disponible (streaming mot par mot), au lieu
# d'un seul bloc qui n'apparaît qu'une fois les deux générations terminées.
import gradio as gr

PLACEHOLDER = "*(en attente)*"

def ui_fn(question, language, compare):
    if not question or not question.strip():
        yield "Posez une question ci-dessus, puis cliquez sur Générer.", ""
        return

    prompt = build_prompt(question.strip(), language)
    chat_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True,
    )

    # Retour immédiat : indispensable, sinon rien ne bouge à l'écran pendant les premières secondes
    # (chargement du prompt, lancement du thread de génération) et le bouton semble ne rien faire.
    yield "⏳ **Gemma fine-tuné** — génération en cours...", (PLACEHOLDER if compare else "")

    t0 = time.time()
    partial_ft = ""
    for partial_ft in stream_one(chat_prompt):
        yield f"⏳ **Gemma fine-tuné**\n\n{partial_ft}▌", (PLACEHOLDER if compare else "")
    t_ft = time.time() - t0
    out_ft = f"**Gemma fine-tuné** — généré en {t_ft:.1f} s\n\n{partial_ft}"

    if not compare:
        yield out_ft, ""
        return

    yield out_ft, "⏳ **Gemma zero-shot (avant fine-tuning)** — génération en cours..."

    t1 = time.time()
    partial_zs = ""
    with model.disable_adapter():  # désactive temporairement LoRA : on retrouve le modèle de base
        for partial_zs in stream_one(chat_prompt):
            yield out_ft, f"⏳ **Gemma zero-shot (avant fine-tuning)**\n\n{partial_zs}▌"
    t_zs = time.time() - t1
    out_zs = f"**Gemma zero-shot (avant fine-tuning)** — généré en {t_zs:.1f} s\n\n{partial_zs}"
    yield out_ft, out_zs


CUSTOM_CSS = """
.gradio-container { max-width: 980px !important; margin: 0 auto; }
#title-md h2 { margin-bottom: 4px; }
#subtitle-md p { color: #6b7280; margin-top: 0; }
.output-panel {
    border: 1px solid #e2ddd0;
    border-radius: 10px;
    padding: 16px 18px;
    background: #fdfcf9;
    min-height: 120px;
}
#panel-finetuned { border-left: 4px solid #1f5e5b; }
#panel-zeroshot  { border-left: 4px solid #b8862e; }
#generate-btn { font-size: 1.05rem; height: 46px; }
"""

with gr.Blocks(title="Démo Gemma — QA santé multilingue", theme=gr.themes.Soft(primary_hue="teal"), css=CUSTOM_CSS) as demo:
    gr.Markdown("## Démo en direct — Gemma E2B + LoRA", elem_id="title-md")
    gr.Markdown(
        "Question de santé dans n'importe laquelle des 5 langues, réponse générée en direct par le modèle fine-tuné.",
        elem_id="subtitle-md",
    )
    with gr.Group():
        with gr.Row():
            question_box = gr.Textbox(
                label="Question de santé", placeholder="Ex. : Malaria ni nini?", lines=2, scale=3,
            )
            language_dd = gr.Dropdown(choices=LANGUAGES, value="English", label="Langue de réponse", scale=1)
        compare_cb = gr.Checkbox(label="Comparer avec la version avant fine-tuning (zero-shot)", value=False)
        submit_btn = gr.Button("Générer", variant="primary", elem_id="generate-btn")

    with gr.Row():
        out_ft_md = gr.Markdown(value=PLACEHOLDER, elem_id="panel-finetuned", elem_classes=["output-panel"])
        out_zs_md = gr.Markdown(value="", elem_id="panel-zeroshot", elem_classes=["output-panel"])

    submit_btn.click(ui_fn, inputs=[question_box, language_dd, compare_cb], outputs=[out_ft_md, out_zs_md])

# share=True : lien public temporaire (~72h), utilisable par le jury depuis son propre appareil pendant
# la soutenance. Lancer cette cellule 10-15 min avant de présenter (temps de chargement du modèle déjà fait
# plus haut ; ce lancement lui-même est quasi instantané).
demo.launch(share=True, debug=False)

---
**Pour la soutenance:** lancer toutes les cellules à l'avance, garder cet onglet Colab ouvert pendant la présentation (le lien `share=True` meurt si la session s'arrête). Prévoir une courte vidéo de secours de la démo qui fonctionne, au cas où le Wi-Fi ou le quota GPU lâche au mauvais moment.

**Amélioration optionnelle plus tard:** une fois `06_merge_adapter.ipynb` terminé, remplacer le bloc de chargement (base 4-bit + `PeftModel` + déballage) par un chargement direct depuis `checkpoints/gemma-4-e2b-merged` — plus rapide, sans dépendance à Hugging Face. La comparaison zero-shot perdrait alors son mécanisme actuel (`disable_adapter`) et demanderait de charger le modèle de base en plus, si vous voulez la garder.